# PeMS 站点连接关系 LLM 批量标注

## 工作流
1. 准备数据：按高速+方向分组
2. 生成 Prompt：包含站点列表和上下文
3. 调用 LLM：批量推断连接关系
4. 解析结果：提取结构化连接数据
5. 验证修正：检查逻辑一致性

In [ ]:
import pandas as pd
import json
import os
from anthropic import Anthropic

# ============== 配置 ==============

# PeMS 修正后的元数据
PEMS_META = "./output/d3_interpolation_correction/pems_d3_meta_corrected.csv"

# Caltrans 出口数据（可选）
CALTRANS_EXITS = None  # 如果有的话

# 输出目录
OUTPUT_DIR = "./output/llm_annotation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# API 配置
client = Anthropic()

print("配置完成！")

## 1. Prompt 模板设计

In [ ]:
SYSTEM_PROMPT = """
你是一个交通工程专家，专门分析加州高速公路检测站点的拓扑连接关系。

## 背景知识

### 站点类型
- **ML (Mainline)**: 主线站点，监测高速主车道流量
- **HV (HOV)**: HOV车道站点，监测高速拼车道流量
- **OR (On-Ramp)**: 入口匝道站点，监测进入高速的流量
- **FR (Off-Ramp)**: 出口匝道站点，监测离开高速的流量
- **FF (Freeway-Freeway)**: 立交连接器，监测高速之间的连接流量

### 流量方向规则
- 车辆沿 Abs_PM 递增方向行驶（N/E 方向）
- OR：流量从地面道路**进入**高速主线
- FR：流量从高速主线**离开**到地面道路
- FF：流量从一条高速**转移**到另一条高速

### Name 字段常见缩写
- JSO = Just South Of（就在...南边）
- JNO = Just North Of（就在...北边）
- JEO = Just East Of（就在...东边）
- JWO = Just West Of（就在...西边）
- NB/SB/EB/WB = 北向/南向/东向/西向
- Jct = Junction（立交）

### 连接规则
1. ML 站点之间按 PM 顺序串联
2. OR 站点的上游是地面道路（用 Name 推断），下游连接最近的 ML
3. FR 站点的上游是最近的 ML，下游是地面道路（用 Name 推断）
4. HV 通常与 ML 并行，可能有进出连接
5. FF 根据 Name 判断来源和目标高速

## 输出格式

请以 JSON 格式输出每个站点的连接关系：

```json
{
  "segment": "99N PM 10-20",
  "stations": [
    {
      "id": "站点ID",
      "type": "ML/HV/OR/FR/FF",
      "name": "站点名称",
      "abs_pm": 12.5,
      "upstream": ["上游站点ID或描述"],
      "downstream": ["下游站点ID或描述"],
      "flow_direction": "in/out/through/transfer",
      "connected_road": "连接的地面道路或高速（如有）",
      "notes": "备注说明"
    }
  ],
  "edges": [
    {
      "from": "起点站点ID",
      "to": "终点站点ID",
      "type": "mainline/on_ramp/off_ramp/hov/connector",
      "notes": "备注"
    }
  ]
}
```
"""

print("System Prompt 定义完成")

In [ ]:
def generate_segment_prompt(segment_stations, fwy, direction, caltrans_exits=None):
    """
    为一个路段生成标注 Prompt
    
    参数:
        segment_stations: 路段内的站点 DataFrame
        fwy: 高速编号
        direction: 方向 (N/S/E/W)
        caltrans_exits: Caltrans 出口数据（可选）
    """
    # 按 PM 排序
    stations = segment_stations.sort_values('Abs_PM').reset_index(drop=True)
    
    pm_range = f"{stations['Abs_PM'].min():.1f}-{stations['Abs_PM'].max():.1f}"
    
    # 构建站点列表
    station_list = []
    for _, row in stations.iterrows():
        station_list.append({
            'id': row['ID'],
            'type': row['Type'],
            'name': row.get('Name', ''),
            'abs_pm': round(row['Abs_PM'], 3),
            'lat': round(row['Latitude'], 6),
            'lon': round(row['Longitude'], 6),
        })
    
    # 构建 Prompt
    prompt = f"""
## 任务

分析以下高速公路路段的站点连接关系。

## 路段信息

- 高速: {fwy}
- 方向: {direction} ({'北向' if direction == 'N' else '南向' if direction == 'S' else '东向' if direction == 'E' else '西向'})
- PM 范围: {pm_range}
- 站点数: {len(stations)}

## 站点列表

按 Abs_PM 升序排列（车辆行驶方向）：

```json
{json.dumps(station_list, indent=2, ensure_ascii=False)}
```

## 要求

1. 分析每个站点的上下游连接关系
2. 根据 Name 字段推断站点连接的地面道路或其他高速
3. 构建站点之间的边（edge）列表
4. 输出 JSON 格式结果

请特别注意：
- OR 站点：识别它连接的是哪条地面道路（从 Name 推断）
- FR 站点：识别它通向哪条地面道路
- FF 站点：识别它连接的是哪两条高速
- 相邻 ML 站点之间的主线连接
"""
    
    # 如果有 Caltrans 出口数据，添加参考
    if caltrans_exits is not None:
        # 筛选该路段的出口
        pm_min, pm_max = stations['Abs_PM'].min(), stations['Abs_PM'].max()
        relevant_exits = caltrans_exits[
            (caltrans_exits['Route'] == int(fwy)) &
            (caltrans_exits['PM'] >= pm_min - 1) &
            (caltrans_exits['PM'] <= pm_max + 1)
        ]
        
        if len(relevant_exits) > 0:
            prompt += f"""

## Caltrans 官方出口参考

```
{relevant_exits[['Exit', 'PM', 'Name']].to_string(index=False)}
```
"""
    
    return prompt


print("Prompt 生成函数定义完成")

## 2. 数据准备

In [ ]:
# 加载 PeMS 数据
pems_df = pd.read_csv(PEMS_META, dtype={'ID': str, 'Fwy': str})
print(f"总站点数: {len(pems_df)}")
print(f"\n类型分布:")
print(pems_df['Type'].value_counts())
print(f"\n高速分布:")
print(pems_df['Fwy'].value_counts())

In [ ]:
def split_into_segments(df, max_stations_per_segment=15):
    """
    将数据按高速+方向分组，并拆分为合适大小的段
    
    每段不超过 max_stations_per_segment 个站点，
    以避免 Prompt 过长
    """
    segments = []
    
    for (fwy, direction), group in df.groupby(['Fwy', 'Dir']):
        # 按 PM 排序
        sorted_group = group.sort_values('Abs_PM').reset_index(drop=True)
        
        # 拆分为多个段
        n_stations = len(sorted_group)
        n_segments = (n_stations + max_stations_per_segment - 1) // max_stations_per_segment
        
        for i in range(n_segments):
            start_idx = i * max_stations_per_segment
            # 添加重叠：包含前一段的最后2个站点（用于连接）
            if i > 0:
                start_idx = max(0, start_idx - 2)
            end_idx = min((i + 1) * max_stations_per_segment, n_stations)
            
            segment_df = sorted_group.iloc[start_idx:end_idx].copy()
            
            pm_min = segment_df['Abs_PM'].min()
            pm_max = segment_df['Abs_PM'].max()
            
            segments.append({
                'fwy': fwy,
                'direction': direction,
                'segment_id': f"{fwy}{direction}_{i+1}",
                'pm_range': f"{pm_min:.1f}-{pm_max:.1f}",
                'n_stations': len(segment_df),
                'data': segment_df,
            })
    
    return segments


segments = split_into_segments(pems_df)
print(f"共拆分为 {len(segments)} 个段")

# 显示前几个段
for seg in segments[:5]:
    print(f"  {seg['segment_id']}: PM {seg['pm_range']}, {seg['n_stations']} 站点")

## 3. LLM 调用

In [ ]:
def call_llm(prompt, system_prompt=SYSTEM_PROMPT, model="claude-sonnet-4-20250514"):
    """
    调用 Claude API 进行标注
    """
    try:
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=system_prompt,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        return response.content[0].text
    
    except Exception as e:
        print(f"API 调用失败: {e}")
        return None


def parse_llm_response(response_text):
    """
    解析 LLM 返回的 JSON
    """
    try:
        # 提取 JSON 部分
        import re
        json_match = re.search(r'```json\s*(.+?)\s*```', response_text, re.DOTALL)
        
        if json_match:
            json_str = json_match.group(1)
        else:
            # 尝试直接解析
            json_str = response_text
        
        return json.loads(json_str)
    
    except json.JSONDecodeError as e:
        print(f"JSON 解析失败: {e}")
        return None


print("LLM 调用函数定义完成")

In [ ]:
def process_segment(segment, caltrans_exits=None, save_intermediate=True):
    """
    处理单个路段
    """
    segment_id = segment['segment_id']
    print(f"处理 {segment_id}...")
    
    # 生成 Prompt
    prompt = generate_segment_prompt(
        segment['data'],
        segment['fwy'],
        segment['direction'],
        caltrans_exits
    )
    
    # 调用 LLM
    response = call_llm(prompt)
    
    if response is None:
        return None
    
    # 解析结果
    result = parse_llm_response(response)
    
    # 保存中间结果
    if save_intermediate and result:
        with open(f"{OUTPUT_DIR}/{segment_id}.json", 'w') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
    
    return result


print("段处理函数定义完成")

In [ ]:
# 测试：处理第一个段
test_segment = segments[0]
print(f"测试段: {test_segment['segment_id']}")
print(f"站点数: {test_segment['n_stations']}")
print(f"\n站点列表:")
print(test_segment['data'][['ID', 'Type', 'Name', 'Abs_PM']].to_string(index=False))

In [ ]:
# 生成测试 Prompt
test_prompt = generate_segment_prompt(
    test_segment['data'],
    test_segment['fwy'],
    test_segment['direction']
)

print("=" * 50)
print("测试 Prompt:")
print("=" * 50)
print(test_prompt)

In [ ]:
# 调用 LLM 测试
# 注意：需要设置 ANTHROPIC_API_KEY 环境变量

# test_result = process_segment(test_segment)
# print(json.dumps(test_result, indent=2, ensure_ascii=False))

## 4. 批量处理

In [ ]:
from tqdm import tqdm
import time

def batch_process(segments, caltrans_exits=None, delay_seconds=1):
    """
    批量处理所有段
    
    参数:
        segments: 段列表
        caltrans_exits: Caltrans 出口数据
        delay_seconds: API 调用间隔（避免限流）
    """
    all_results = []
    failed = []
    
    for segment in tqdm(segments, desc="批量标注"):
        result = process_segment(segment, caltrans_exits)
        
        if result:
            all_results.append({
                'segment_id': segment['segment_id'],
                'result': result
            })
        else:
            failed.append(segment['segment_id'])
        
        # 避免 API 限流
        time.sleep(delay_seconds)
    
    print(f"\n处理完成: {len(all_results)} 成功, {len(failed)} 失败")
    
    if failed:
        print(f"失败的段: {failed}")
    
    return all_results, failed


print("批量处理函数定义完成")

In [ ]:
# 批量处理（取消注释运行）
# all_results, failed = batch_process(segments)

## 5. 结果合并与验证

In [ ]:
def merge_results(all_results):
    """
    合并所有段的结果
    """
    all_stations = []
    all_edges = []
    
    for item in all_results:
        result = item['result']
        segment_id = item['segment_id']
        
        # 提取站点信息
        for station in result.get('stations', []):
            station['segment_id'] = segment_id
            all_stations.append(station)
        
        # 提取边信息
        for edge in result.get('edges', []):
            edge['segment_id'] = segment_id
            all_edges.append(edge)
    
    # 去重（重叠段的站点）
    stations_df = pd.DataFrame(all_stations)
    if 'id' in stations_df.columns:
        stations_df = stations_df.drop_duplicates(subset=['id'], keep='first')
    
    edges_df = pd.DataFrame(all_edges)
    if 'from' in edges_df.columns and 'to' in edges_df.columns:
        edges_df = edges_df.drop_duplicates(subset=['from', 'to'], keep='first')
    
    return stations_df, edges_df


def validate_edges(edges_df, pems_df):
    """
    验证边的逻辑一致性
    """
    issues = []
    
    station_ids = set(pems_df['ID'].values)
    
    for _, edge in edges_df.iterrows():
        from_id = edge['from']
        to_id = edge['to']
        
        # 检查站点是否存在
        if from_id not in station_ids:
            issues.append(f"边 {from_id} -> {to_id}: 起点不存在")
        
        if to_id not in station_ids:
            # 可能是地面道路，不是问题
            pass
        
        # 检查 PM 方向一致性
        if from_id in station_ids and to_id in station_ids:
            from_pm = pems_df[pems_df['ID'] == from_id]['Abs_PM'].values[0]
            to_pm = pems_df[pems_df['ID'] == to_id]['Abs_PM'].values[0]
            
            # 对于 N/E 方向，PM 应递增
            from_dir = pems_df[pems_df['ID'] == from_id]['Dir'].values[0]
            
            if from_dir in ['N', 'E'] and to_pm < from_pm:
                issues.append(f"边 {from_id} -> {to_id}: PM 方向不一致 ({from_pm} -> {to_pm})")
    
    return issues


print("验证函数定义完成")

## 6. 可视化验证

In [ ]:
def visualize_connections(pems_df, edges_df, fwy, direction, output_path):
    """
    可视化标注的连接关系
    """
    import folium
    
    # 筛选数据
    stations = pems_df[(pems_df['Fwy'] == fwy) & (pems_df['Dir'] == direction)]
    
    if len(stations) == 0:
        print(f"无 {fwy}{direction} 数据")
        return None
    
    center_lat = stations['Latitude'].mean()
    center_lon = stations['Longitude'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=11)
    
    # 站点颜色
    type_colors = {
        'ML': '#E53935',
        'HV': '#8E24AA',
        'OR': '#4CAF50',
        'FR': '#FF9800',
        'FF': '#FFC107',
    }
    
    # 绘制站点
    station_coords = {}
    for _, station in stations.iterrows():
        color = type_colors.get(station['Type'], '#888')
        
        folium.CircleMarker(
            [station['Latitude'], station['Longitude']],
            radius=8,
            color=color,
            fill=True,
            fillOpacity=0.8,
            popup=f"{station['ID']}<br>{station['Type']}<br>{station.get('Name', '')}",
            tooltip=f"{station['ID']} ({station['Type']})"
        ).add_to(m)
        
        station_coords[station['ID']] = (station['Latitude'], station['Longitude'])
    
    # 绘制边
    for _, edge in edges_df.iterrows():
        from_id = edge['from']
        to_id = edge['to']
        
        if from_id in station_coords and to_id in station_coords:
            from_coord = station_coords[from_id]
            to_coord = station_coords[to_id]
            
            # 边颜色
            edge_type = edge.get('type', 'unknown')
            if edge_type == 'mainline':
                color = '#2196F3'
            elif edge_type == 'on_ramp':
                color = '#4CAF50'
            elif edge_type == 'off_ramp':
                color = '#FF9800'
            else:
                color = '#888'
            
            folium.PolyLine(
                [from_coord, to_coord],
                color=color,
                weight=3,
                opacity=0.7,
                popup=f"{from_id} → {to_id}<br>{edge_type}"
            ).add_to(m)
    
    m.save(output_path)
    print(f"已保存: {output_path}")
    return m


print("可视化函数定义完成")

## 7. 导出最终结果

In [ ]:
def export_results(stations_df, edges_df, output_dir):
    """
    导出标注结果
    """
    # 站点连接信息
    stations_df.to_csv(f"{output_dir}/annotated_stations.csv", index=False)
    
    # 边列表
    edges_df.to_csv(f"{output_dir}/annotated_edges.csv", index=False)
    
    # 邻接表格式
    adjacency = edges_df.groupby('from')['to'].apply(list).to_dict()
    with open(f"{output_dir}/adjacency.json", 'w') as f:
        json.dump(adjacency, f, indent=2)
    
    print(f"导出完成:")
    print(f"  - {output_dir}/annotated_stations.csv")
    print(f"  - {output_dir}/annotated_edges.csv")
    print(f"  - {output_dir}/adjacency.json")


print("导出函数定义完成")

## 总结

### 工作流程

```
1. 加载 PeMS 元数据
2. 按高速+方向分组，拆分为小段（每段 ~15 站点）
3. 为每段生成 Prompt
4. 调用 Claude API 进行标注
5. 解析 JSON 结果
6. 合并去重
7. 验证逻辑一致性
8. 可视化检查
9. 导出最终结果
```

### 输出文件

| 文件 | 说明 |
|------|------|
| `annotated_stations.csv` | 站点连接信息 |
| `annotated_edges.csv` | 边列表 |
| `adjacency.json` | 邻接表 |
| `{segment_id}.json` | 每段的中间结果 |

### 使用方式

```python
# 1. 设置 API Key
export ANTHROPIC_API_KEY="your-key"

# 2. 运行批量处理
all_results, failed = batch_process(segments)

# 3. 合并验证
stations_df, edges_df = merge_results(all_results)
issues = validate_edges(edges_df, pems_df)

# 4. 导出
export_results(stations_df, edges_df, OUTPUT_DIR)
```